# POC — Coleta de Dados do YouTube (Ceará 2026)

Teste inicial de ingestão via **YouTube Data API v3**, restrito ao escopo da prova de conceito definido em [`fontes_youtube.md`](../fontes_youtube.md):

- **Candidatos:** Ciro Gomes (PSDB) e Elmano de Freitas (PT)
- **Imprensa:** O Povo e Diário do Nordeste

Objetivo deste notebook: validar que conseguimos acessar a API, resolver os canais, listar vídeos recentes e puxar estatísticas básicas (views, likes, nº de comentários) — servindo também como **prova de acesso à fonte** (item 8 da Entrega 1).

Este notebook **não coleta comentários ainda** — isso fica para um próximo passo, depois de validar a coleta de metadados de vídeo.

## 0. Configuração

Antes de rodar: copie `.env.example` para `.env` na raiz do projeto e preencha `YOUTUBE_API_KEY` com uma chave de API do Google Cloud Console (API "YouTube Data API v3" habilitada). O `.env` já está no `.gitignore` — nunca comitar a chave.

In [ ]:
%pip install --quiet google-api-python-client python-dotenv pandas

In [ ]:
import os
import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from googleapiclient.discovery import build

load_dotenv()

API_KEY = os.environ.get("YOUTUBE_API_KEY")
if not API_KEY:
    raise RuntimeError(
        "YOUTUBE_API_KEY não encontrada. Crie um arquivo .env na raiz do projeto "
        "(veja .env.example) com sua chave da YouTube Data API v3."
    )

youtube = build("youtube", "v3", developerKey=API_KEY)
print("Cliente da YouTube Data API criado com sucesso.")

## 1. Canais da POC

Handles/URLs identificados em `fontes_youtube.md`. Alguns são handles (`@...`), outros ainda usam o formato legado `c/NomeDoCanal` — para esses últimos, resolvemos por busca do nome exato (fallback), já que `forHandle` só funciona com handles `@`.

In [ ]:
POC_CHANNELS = {
    # "ciro_gomes": {"handle": "CanalCirooGomes", "categoria": "candidato"},
    # "elmano_de_freitas": {"handle": "ElmanodeFreitas", "categoria": "candidato"},
    # "o_povo": {"handle": "opovo", "categoria": "imprensa"},
    "diario_do_nordeste": {"handle": "diariodonordeste", "categoria": "imprensa"},
}
POC_CHANNELS

## 2. Resolver `channelId` a partir do handle

A API precisa do `channelId` (formato `UC...`) para as próximas chamadas. Usamos `channels.list(part="id,snippet,contentDetails", forHandle=...)`, que também já retorna o `uploads` playlist ID em `contentDetails.relatedPlaylists.uploads` — economizando uma chamada.

In [ ]:
def resolve_channel(handle: str) -> dict | None:
    """Resolve channelId e uploads playlist a partir de um handle (@...)."""
    response = (
        youtube.channels()
        .list(part="id,snippet,contentDetails", forHandle=handle)
        .execute()
    )
    items = response.get("items", [])
    if not items:
        return None
    item = items[0]
    return {
        "channel_id": item["id"],
        "title": item["snippet"]["title"],
        "uploads_playlist_id": item["contentDetails"]["relatedPlaylists"]["uploads"],
    }


for key, info in POC_CHANNELS.items():
    resolved = resolve_channel(info["handle"])
    if resolved is None:
        print(f"[AVISO] Não foi possível resolver o handle @{info['handle']} ({key}). Verifique manualmente.")
        continue
    info.update(resolved)
    print(f"{key}: {resolved['title']} -> {resolved['channel_id']}")

## 3. Listar vídeos recentes de cada canal

Usa `playlistItems.list` sobre a *uploads playlist* de cada canal (1 unidade de cota por página, até 50 vídeos) — bem mais barato que `search.list`.

In [ ]:
def list_recent_videos(uploads_playlist_id: str, max_results: int = 15) -> list[dict]:
    """Lista os vídeos mais recentes de uma uploads playlist."""
    response = (
        youtube.playlistItems()
        .list(part="snippet,contentDetails", playlistId=uploads_playlist_id, maxResults=max_results)
        .execute()
    )
    videos = []
    for item in response.get("items", []):
        videos.append(
            {
                "video_id": item["contentDetails"]["videoId"],
                "title": item["snippet"]["title"],
                "published_at": item["contentDetails"].get("videoPublishedAt"),
            }
        )
    return videos


for key, info in POC_CHANNELS.items():
    if "uploads_playlist_id" not in info:
        continue
    videos = list_recent_videos(info["uploads_playlist_id"])
    info["recent_videos"] = videos
    print(f"{key}: {len(videos)} vídeos encontrados")

## 4. Estatísticas dos vídeos (views, likes, nº de comentários)

`videos.list` aceita até 50 IDs por chamada (1 unidade de cota), então buscamos as estatísticas de todos os vídeos de um canal em uma única chamada.

In [ ]:
def get_video_stats(video_ids: list[str]) -> dict[str, dict]:
    """Busca estatísticas (views, likes, comentários) para uma lista de video IDs."""
    if not video_ids:
        return {}
    response = (
        youtube.videos()
        .list(part="statistics,snippet", id=",".join(video_ids))
        .execute()
    )
    stats = {}
    for item in response.get("items", []):
        stats[item["id"]] = {
            "views": int(item["statistics"].get("viewCount", 0)),
            "likes": int(item["statistics"].get("likeCount", 0)),
            "comment_count": int(item["statistics"].get("commentCount", 0)),
            "channel_title": item["snippet"]["channelTitle"],
        }
    return stats


all_rows = []
for key, info in POC_CHANNELS.items():
    videos = info.get("recent_videos", [])
    stats = get_video_stats([v["video_id"] for v in videos])
    for v in videos:
        row = {"source_key": key, **v, **stats.get(v["video_id"], {})}
        all_rows.append(row)

df = pd.DataFrame(all_rows)
df.sort_values(["source_key", "published_at"], ascending=[True, False], inplace=True)
df

## 5. Salvar amostra bruta (prova de acesso à fonte)

Grava o resultado bruto em `data/raw/youtube/` com timestamp — serve como evidência de que a extração funcionou (item 8 da Entrega 1) e como ponto de partida da camada Bronze.

In [ ]:
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
output_dir = Path("../data/raw/youtube") / timestamp
output_dir.mkdir(parents=True, exist_ok=True)

for key, info in POC_CHANNELS.items():
    payload = {
        "collected_at": timestamp,
        "source_key": key,
        "channel_id": info.get("channel_id"),
        "channel_title": info.get("title"),
        "categoria": info.get("categoria"),
        "videos": [
            {**v, **get_video_stats([v["video_id"]]).get(v["video_id"], {})}
            for v in info.get("recent_videos", [])
        ],
    }
    out_path = output_dir / f"{key}.json"
    out_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Salvo: {out_path}")

## 6. Resumo da coleta

Conferência rápida: quantos vídeos e qual o total de comentários (nível 1) disponíveis por canal — dá uma ideia do volume real, útil para estimar cota necessária na coleta de comentários (próximo passo).

In [ ]:
summary = (
    df.groupby("source_key")
    .agg(videos=("video_id", "count"), views=("views", "sum"), likes=("likes", "sum"), comentarios=("comment_count", "sum"))
    .reset_index()
)
summary

## Próximos passos

- Validar manualmente os `channel_id` resolvidos (conferir se batem com os canais esperados).
- Se algum canal não resolver via `forHandle` (ex.: URLs legadas `c/NomeDoCanal`), buscar o `channelId` manualmente pela página do canal (`Ver código-fonte` → `channelId`) e adicionar direto no dicionário `POC_CHANNELS`.
- Próximo notebook/script: coletar comentários (`commentThreads.list` + `comments.list`) dos vídeos listados aqui, já usando os `video_id` salvos em `data/raw/youtube/`.
- Depois de validar a POC, mover esta lógica para os scripts do pipeline (Airflow) em vez de notebook.